# Анализ данных персонажей D&D 5e: кластеризация и классификация
## Дипломная работа: ИС поддержки боевых столкновений НРИ

**Студент:** Павлов Михаил Владимирович  
**Направление:** Прикладная информатика  

---

### Описание задачи

В данном ноутбуке решаются две задачи машинного обучения на данных персонажей и NPC из системы D&D 5e:

**Задача 1 — Кластеризация (обучение без учителя):**  
Разбить персонажей на группы по их характеристикам без использования меток классов.  
Методы: K-Means, DBSCAN, AgglomerativeClustering.

**Задача 2 — Классификация (обучение с учителем):**  
Предсказать уровень сложности (difficulty) противника по его параметрам.  
Методы: DecisionTreeClassifier, GaussianNB, RandomForestClassifier.

---
## 1. Подготовка: импорты и генерация данных

In [ ]:
# ── Стандартные библиотеки ──────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns

# ── Scikit-learn: предобработка и оценка ────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# ── Scikit-learn: кластеризация ──────────────────────────────────────────
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering

# ── Scikit-learn: классификация ──────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# ── Scipy для дендрограммы ───────────────────────────────────────────────
from scipy.cluster.hierarchy import dendrogram, linkage

# Настройки визуализации
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ Все библиотеки успешно импортированы')
print(f'   NumPy: {np.__version__}')
print(f'   Pandas: {pd.__version__}')
import sklearn; print(f'   Scikit-learn: {sklearn.__version__}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Генерация синтетического датасета персонажей D&D 5e
# ══════════════════════════════════════════════════════════════════════════
#
# Параметры персонажей:
#   HP           — очки здоровья (Hit Points)
#   AC           — класс брони (Armor Class)
#   STR, DEX,
#   CON, INT,
#   WIS, CHA     — шесть основных характеристик (3–20)
#   CR           — Challenge Rating (уровень сложности врага)
#   attack_bonus — бонус к атаке
#   damage_avg   — средний урон за раунд
#   difficulty   — категориальная метка сложности (целевая переменная)

n = 300  # количество персонажей/NPC

# Три архетипа врагов для реалистичной кластеризации
# Архетип 1: «рядовые» (мелкие монстры — гоблины, крысы)
n1 = 100
arch1 = {
    'HP':  np.random.normal(18, 4, n1).clip(5, 40).astype(int),
    'AC':  np.random.normal(13, 1.5, n1).clip(10, 17).astype(int),
    'STR': np.random.normal(10, 2, n1).clip(4, 16).astype(int),
    'DEX': np.random.normal(14, 2, n1).clip(6, 18).astype(int),
    'CON': np.random.normal(10, 2, n1).clip(4, 16).astype(int),
    'INT': np.random.normal(8,  2, n1).clip(3, 13).astype(int),
    'WIS': np.random.normal(8,  2, n1).clip(3, 13).astype(int),
    'CHA': np.random.normal(7,  2, n1).clip(3, 12).astype(int),
    'CR':  np.random.choice([0.125, 0.25, 0.5, 1], n1, p=[0.3, 0.3, 0.3, 0.1]),
    'attack_bonus': np.random.normal(3, 1, n1).clip(1, 5).astype(int),
    'damage_avg':   np.random.normal(5, 2, n1).clip(2, 12).astype(int),
}

# Архетип 2: «элитные» (офицеры, оборотни, нежить)
n2 = 120
arch2 = {
    'HP':  np.random.normal(65, 15, n2).clip(30, 110).astype(int),
    'AC':  np.random.normal(15, 1.5, n2).clip(12, 19).astype(int),
    'STR': np.random.normal(16, 2,  n2).clip(10, 22).astype(int),
    'DEX': np.random.normal(12, 2,  n2).clip(6,  18).astype(int),
    'CON': np.random.normal(16, 2,  n2).clip(10, 22).astype(int),
    'INT': np.random.normal(12, 3,  n2).clip(5,  19).astype(int),
    'WIS': np.random.normal(12, 2,  n2).clip(5,  18).astype(int),
    'CHA': np.random.normal(11, 2,  n2).clip(5,  17).astype(int),
    'CR':  np.random.choice([2, 3, 4, 5], n2, p=[0.3, 0.3, 0.25, 0.15]),
    'attack_bonus': np.random.normal(6, 1.5, n2).clip(3, 10).astype(int),
    'damage_avg':   np.random.normal(18, 5,  n2).clip(8, 35).astype(int),
}

# Архетип 3: «боссы» (драконы, великаны, архидемоны)
n3 = 80
arch3 = {
    'HP':  np.random.normal(200, 40, n3).clip(120, 330).astype(int),
    'AC':  np.random.normal(18, 2,  n3).clip(15, 24).astype(int),
    'STR': np.random.normal(22, 3,  n3).clip(14, 30).astype(int),
    'DEX': np.random.normal(10, 2,  n3).clip(5,  16).astype(int),
    'CON': np.random.normal(22, 3,  n3).clip(14, 30).astype(int),
    'INT': np.random.normal(16, 3,  n3).clip(8,  25).astype(int),
    'WIS': np.random.normal(14, 2,  n3).clip(8,  20).astype(int),
    'CHA': np.random.normal(18, 3,  n3).clip(10, 26).astype(int),
    'CR':  np.random.choice([10, 13, 15, 17, 20, 24], n3,
                             p=[0.25, 0.2, 0.2, 0.15, 0.1, 0.1]),
    'attack_bonus': np.random.normal(12, 2, n3).clip(8, 17).astype(int),
    'damage_avg':   np.random.normal(65, 15, n3).clip(35, 110).astype(int),
}

# Собираем DataFrame
frames = []
for arch, label in [(arch1, 'Easy'), (arch2, 'Medium'), (arch3, 'Hard')]:
    df_arch = pd.DataFrame(arch)
    df_arch['difficulty'] = label
    frames.append(df_arch)

df = pd.concat(frames, ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Датасет: {df.shape[0]} строк × {df.shape[1]} столбцов')
print('\nРаспределение классов:')
print(df['difficulty'].value_counts())
df.head(8)

In [ ]:
# ── Разведочный анализ данных (EDA) ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
features = ['HP', 'AC', 'STR', 'DEX', 'CON', 'INT', 'WIS', 'CHA']
colors = {'Easy': '#4CAF50', 'Medium': '#FF9800', 'Hard': '#F44336'}

for ax, feat in zip(axes.flat, features):
    for diff, color in colors.items():
        subset = df[df['difficulty'] == diff][feat]
        ax.hist(subset, bins=20, alpha=0.6, color=color, label=diff)
    ax.set_title(feat, fontsize=13, fontweight='bold')
    ax.set_xlabel('Значение')
    ax.set_ylabel('Частота')
    ax.legend(fontsize=8)

plt.suptitle('Распределение характеристик персонажей по категориям сложности',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('📊 Гистограммы построены')

In [ ]:
# ── Предобработка данных ─────────────────────────────────────────────────
FEATURES = ['HP', 'AC', 'STR', 'DEX', 'CON', 'INT', 'WIS', 'CHA',
            'attack_bonus', 'damage_avg']
TARGET   = 'difficulty'

X = df[FEATURES].values
y = df[TARGET].values

# Стандартизация: μ=0, σ=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA до 2 компонент для визуализации
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f'Признаков: {X.shape[1]}')
print(f'Объектов: {X.shape[0]}')
print(f'PCA дисперсия (2 компоненты): '
      f'{pca.explained_variance_ratio_.sum():.1%}')

# Кодируем метки для классификации
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f'Классы: {le.classes_}')

---
## 2. КЛАСТЕРИЗАЦИЯ (обучение без учителя)

| Метод | Описание | Гиперпараметры |
|---|---|---|
| **K-Means** | Минимизирует внутрикластерную дисперсию | `k=3`, `init='k-means++'` |
| **DBSCAN** | Кластеры как плотные области, шум = −1 | `eps=0.5`, `min_samples=5` |
| **Agglomerative** | Иерархическое слияние снизу вверх | `n_clusters=3`, `linkage='ward'` |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 2.1  K-MEANS
# ════════════════════════════════════════════════════════════════════════

# Метод локтя — подбираем оптимальное k
inertia_vals = []
silhouette_vals = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10,
                random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    inertia_vals.append(km.inertia_)
    silhouette_vals.append(silhouette_score(X_scaled, labels))

# Визуализация метода локтя
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertia_vals, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', label='k=3 (оптимум)')
axes[0].set_xlabel('Число кластеров k', fontsize=12)
axes[0].set_ylabel('Инерция (SSE)', fontsize=12)
axes[0].set_title('K-Means: метод локтя', fontsize=13, fontweight='bold')
axes[0].legend()

axes[1].plot(k_range, silhouette_vals, 'rs-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='blue', linestyle='--', label='k=3 (оптимум)')
axes[1].set_xlabel('Число кластеров k', fontsize=12)
axes[1].set_ylabel('Коэффициент силуэта', fontsize=12)
axes[1].set_title('K-Means: коэффициент силуэта', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

# Финальная модель K-Means
kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10,
                random_state=RANDOM_STATE)
km_labels = kmeans.fit_predict(X_scaled)

km_sil = silhouette_score(X_scaled, km_labels)
km_db  = davies_bouldin_score(X_scaled, km_labels)

print(f'\n✅ K-Means (k=3):')
print(f'   Коэффициент силуэта: {km_sil:.4f}  (↑ лучше, макс=1)')
print(f'   Индекс Дэвиса–Болдина: {km_db:.4f}  (↓ лучше, мин=0)')
print(f'   Инерция: {kmeans.inertia_:.1f}')

In [ ]:
# Визуализация K-Means в пространстве PCA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cluster_colors = ['#2196F3', '#FF9800', '#4CAF50']
cmap_clusters = ListedColormap(cluster_colors)

# Слева — истинные метки
diff_colors = {'Easy': '#4CAF50', 'Medium': '#FF9800', 'Hard': '#F44336'}
for diff, color in diff_colors.items():
    mask = y == diff
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, label=diff, alpha=0.7, s=40, edgecolors='white', linewidths=0.3)
axes[0].set_title('Истинные метки (difficulty)', fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[0].legend()

# Справа — кластеры K-Means
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                           c=km_labels, cmap=cmap_clusters,
                           alpha=0.7, s=40, edgecolors='white', linewidths=0.3)
# Центроиды
centers_pca = pca.transform(kmeans.cluster_centers_)
axes[1].scatter(centers_pca[:, 0], centers_pca[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Центроиды')
axes[1].set_title(f'K-Means (k=3)  |  Силуэт={km_sil:.3f}',
                   fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
patches = [mpatches.Patch(color=c, label=f'Кластер {i}') for i, c in enumerate(cluster_colors)]
axes[1].legend(handles=patches + [axes[1].get_legend_handles_labels()[0][-1]])

plt.suptitle('K-Means: сравнение с истинными метками', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 2.2  DBSCAN
# ════════════════════════════════════════════════════════════════════════

dbscan = DBSCAN(eps=1.2, min_samples=8, metric='euclidean')
db_labels = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise_db    = (db_labels == -1).sum()

print(f'✅ DBSCAN (eps=1.2, min_samples=8):')
print(f'   Найдено кластеров: {n_clusters_db}')
print(f'   Шумовых точек:     {n_noise_db} ({n_noise_db/len(db_labels):.1%})')

if n_clusters_db >= 2:
    mask_valid = db_labels != -1
    db_sil = silhouette_score(X_scaled[mask_valid], db_labels[mask_valid])
    print(f'   Коэффициент силуэта (без шума): {db_sil:.4f}')

# Визуализация
fig, ax = plt.subplots(figsize=(9, 6))

unique_labels = sorted(set(db_labels))
palette = plt.cm.tab10(np.linspace(0, 1, max(len(unique_labels), 2)))

for lbl, color in zip(unique_labels, palette):
    mask = db_labels == lbl
    if lbl == -1:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c='gray', marker='x', s=40, alpha=0.5, label='Шум')
    else:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=[color], alpha=0.7, s=40,
                   edgecolors='white', linewidths=0.3,
                   label=f'Кластер {lbl}')

ax.set_title(f'DBSCAN (eps=1.2, min_samples=8)\n'
             f'Кластеров: {n_clusters_db}, Шума: {n_noise_db}',
             fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 2.3  ИЕРАРХИЧЕСКАЯ КЛАСТЕРИЗАЦИЯ (AgglomerativeClustering)
# ════════════════════════════════════════════════════════════════════════

# Дендрограмма (на подвыборке для читаемости)
sample_idx = np.random.choice(len(X_scaled), size=80, replace=False)
X_sample   = X_scaled[sample_idx]
y_sample   = y[sample_idx]

linked = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(16, 6))
dendrogram(
    linked,
    labels=y_sample,
    ax=ax,
    leaf_rotation=90,
    leaf_font_size=8,
    color_threshold=linked[-3, 2]
)
ax.set_title('Дендрограмма иерархической кластеризации (метод Ward, подвыборка 80)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Расстояние (Ward)')
ax.axhline(y=linked[-3, 2], color='red', linestyle='--', label='Срез на k=3')
ax.legend()
plt.tight_layout()
plt.show()

# Финальная модель
agglo = AgglomerativeClustering(n_clusters=3, linkage='ward')
ag_labels = agglo.fit_predict(X_scaled)

ag_sil = silhouette_score(X_scaled, ag_labels)
ag_db  = davies_bouldin_score(X_scaled, ag_labels)

print(f'✅ AgglomerativeClustering (k=3, ward):')
print(f'   Коэффициент силуэта:    {ag_sil:.4f}')
print(f'   Индекс Дэвиса–Болдина:  {ag_db:.4f}')

# Визуализация в PCA
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=ag_labels, cmap='Set1',
                     alpha=0.7, s=40, edgecolors='white', linewidths=0.3)
ax.set_title(f'Иерархическая кластеризация (Ward, k=3)\nСилуэт={ag_sil:.3f}',
             fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.colorbar(scatter, ax=ax, label='Кластер')
plt.tight_layout()
plt.show()

In [ ]:
# ── Сводная таблица метрик кластеризации ────────────────────────────────
results_cl = pd.DataFrame([
    {'Метод': 'K-Means (k=3)',
     'Кластеров': 3,
     'Коэфф. силуэта ↑': round(km_sil, 4),
     'Индекс Дэвиса–Болдина ↓': round(km_db, 4),
     'Параметры': 'k=3, init=k-means++'},
    {'Метод': 'DBSCAN',
     'Кластеров': n_clusters_db,
     'Коэфф. силуэта ↑': round(db_sil, 4) if n_clusters_db >= 2 else 'N/A',
     'Индекс Дэвиса–Болдина ↓': '—',
     'Параметры': 'eps=1.2, min_samples=8'},
    {'Метод': 'AgglomerativeClustering',
     'Кластеров': 3,
     'Коэфф. силуэта ↑': round(ag_sil, 4),
     'Индекс Дэвиса–Болдина ↓': round(ag_db, 4),
     'Параметры': 'n_clusters=3, linkage=ward'},
])

print('=' * 70)
print('СВОДНАЯ ТАБЛИЦА: МЕТРИКИ КАЧЕСТВА КЛАСТЕРИЗАЦИИ')
print('=' * 70)
print(results_cl.to_string(index=False))
print('\nЛегенда:')
print('  Силуэт ↑ — чем ближе к 1, тем лучше разделение кластеров')
print('  DB ↓     — чем ближе к 0, тем более компактные и разделённые кластеры')

---
## 3. КЛАССИФИКАЦИЯ (обучение с учителем)

**Задача:** предсказать категорию сложности противника (`Easy` / `Medium` / `Hard`).

| Метод | Описание | Гиперпараметры |
|---|---|---|
| **Decision Tree** | Рекурсивное разбиение пространства признаков | `max_depth=5`, `criterion='gini'` |
| **Naive Bayes** | Вероятностный классификатор на основе теоремы Байеса | `var_smoothing=1e-9` |
| **Random Forest** | Ансамбль деревьев решений | `n_estimators=100`, `max_depth=10` |

In [ ]:
# ── Разделение на обучающую и тестовую выборки ───────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print(f'Обучающая выборка:  {X_train.shape[0]} объектов ({X_train.shape[0]/len(X_scaled):.0%})')
print(f'Тестовая выборка:   {X_test.shape[0]} объектов ({X_test.shape[0]/len(X_scaled):.0%})')
print(f'Признаков:          {X_train.shape[1]}')
print(f'Классы (закодированы): {list(le.classes_)} → {list(range(len(le.classes_)))}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 3.1  ДЕРЕВО РЕШЕНИЙ
# ════════════════════════════════════════════════════════════════════════

dt = DecisionTreeClassifier(
    max_depth=5,
    criterion='gini',
    min_samples_split=10,
    random_state=RANDOM_STATE
)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

dt_acc  = accuracy_score(y_test, dt_pred)
dt_cv   = cross_val_score(dt, X_scaled, y_encoded, cv=5, scoring='accuracy')

print(f'✅ Decision Tree (max_depth=5):')
print(f'   Точность на тесте:   {dt_acc:.4f} ({dt_acc:.1%})')
print(f'   CV-5 точность:       {dt_cv.mean():.4f} ± {dt_cv.std():.4f}')
print()
print(classification_report(y_test, dt_pred, target_names=le.classes_))

# Матрица ошибок
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, dt_pred),
    display_labels=le.classes_
).plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title(f'Дерево решений — матрица ошибок\nТочность={dt_acc:.1%}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Визуализация дерева решений
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt, ax=ax,
    feature_names=FEATURES,
    class_names=le.classes_,
    filled=True,
    rounded=True,
    max_depth=3,  # показываем первые 3 уровня для читаемости
    fontsize=9,
    impurity=False
)
ax.set_title('Дерево решений (первые 3 уровня из 5)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 3.2  НАИВНЫЙ БАЙЕС (GaussianNB)
# ════════════════════════════════════════════════════════════════════════

gnb = GaussianNB()
gnb.fit(X_train, y_train)
gnb_pred = gnb.predict(X_test)

gnb_acc = accuracy_score(y_test, gnb_pred)
gnb_cv  = cross_val_score(gnb, X_scaled, y_encoded, cv=5, scoring='accuracy')

print(f'✅ Gaussian Naive Bayes:')
print(f'   Точность на тесте:   {gnb_acc:.4f} ({gnb_acc:.1%})')
print(f'   CV-5 точность:       {gnb_cv.mean():.4f} ± {gnb_cv.std():.4f}')
print()
print(classification_report(y_test, gnb_pred, target_names=le.classes_))

# Матрица ошибок
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, gnb_pred),
    display_labels=le.classes_
).plot(ax=ax, colorbar=True, cmap='Greens')
ax.set_title(f'Наивный Байес — матрица ошибок\nТочность={gnb_acc:.1%}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Апостериорные вероятности для нескольких объектов
proba_sample = gnb.predict_proba(X_test[:5])
proba_df = pd.DataFrame(proba_sample, columns=le.classes_)
proba_df['Предсказание'] = le.classes_[gnb_pred[:5]]
proba_df['Истина']       = le.classes_[y_test[:5]]
print('\nАпостериорные вероятности (первые 5 объектов теста):')
print(proba_df.round(3).to_string(index=False))

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 3.3  СЛУЧАЙНЫЙ ЛЕС (RandomForestClassifier)
# ════════════════════════════════════════════════════════════════════════

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
rf_cv  = cross_val_score(rf, X_scaled, y_encoded, cv=5, scoring='accuracy')

print(f'✅ Random Forest (100 деревьев, max_depth=10):')
print(f'   Точность на тесте:   {rf_acc:.4f} ({rf_acc:.1%})')
print(f'   CV-5 точность:       {rf_cv.mean():.4f} ± {rf_cv.std():.4f}')
print()
print(classification_report(y_test, rf_pred, target_names=le.classes_))

# Матрица ошибок
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, rf_pred),
    display_labels=le.classes_
).plot(ax=ax, colorbar=True, cmap='Oranges')
ax.set_title(f'Случайный лес — матрица ошибок\nТочность={rf_acc:.1%}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Важность признаков
feat_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
feat_imp.plot(kind='barh', ax=ax, color='#FF9800', edgecolor='black', linewidth=0.5)
ax.set_title('Важность признаков (Random Forest)', fontsize=12, fontweight='bold')
ax.set_xlabel('Важность (Gini impurity decrease)')
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# ИТОГОВОЕ СРАВНЕНИЕ ВСЕХ МЕТОДОВ
# ════════════════════════════════════════════════════════════════════════

print('=' * 65)
print('ИТОГОВОЕ СРАВНЕНИЕ: КЛАСТЕРИЗАЦИЯ')
print('=' * 65)
print(results_cl.to_string(index=False))

results_clf = pd.DataFrame([
    {'Метод': 'Decision Tree',
     'Точность (тест)': f'{dt_acc:.4f}',
     'CV-5 (μ±σ)': f'{dt_cv.mean():.4f}±{dt_cv.std():.4f}',
     'Параметры': 'max_depth=5, gini'},
    {'Метод': 'Naive Bayes',
     'Точность (тест)': f'{gnb_acc:.4f}',
     'CV-5 (μ±σ)': f'{gnb_cv.mean():.4f}±{gnb_cv.std():.4f}',
     'Параметры': 'GaussianNB'},
    {'Метод': 'Random Forest',
     'Точность (тест)': f'{rf_acc:.4f}',
     'CV-5 (μ±σ)': f'{rf_cv.mean():.4f}±{rf_cv.std():.4f}',
     'Параметры': 'n=100, max_depth=10'},
])

print()
print('=' * 65)
print('ИТОГОВОЕ СРАВНЕНИЕ: КЛАССИФИКАЦИЯ')
print('=' * 65)
print(results_clf.to_string(index=False))

# Финальный Bar-chart сравнения классификаторов
methods = ['Decision Tree', 'Naive Bayes', 'Random Forest']
test_accs = [dt_acc, gnb_acc, rf_acc]
cv_means  = [dt_cv.mean(), gnb_cv.mean(), rf_cv.mean()]
cv_stds   = [dt_cv.std(), gnb_cv.std(), rf_cv.std()]

x = np.arange(len(methods))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, test_accs, width, label='Тест', color='#2196F3', alpha=0.85)
bars2 = ax.bar(x + width/2, cv_means,  width, yerr=cv_stds, capsize=5,
               label='CV-5 (μ±σ)', color='#FF9800', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Метод классификации', fontsize=12)
ax.set_ylabel('Точность (Accuracy)', fontsize=12)
ax.set_title('Сравнение методов классификации: тест vs. кросс-валидация',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.axhline(y=1/3, color='gray', linestyle='--', alpha=0.5, label='Baseline (случайный)')
plt.tight_layout()
plt.show()

best_method = methods[np.argmax(test_accs)]
print(f'\n🏆 Наилучший метод: {best_method} (точность {max(test_accs):.1%})')

---
## 4. Выводы

### Кластеризация

| Метод | Вывод |
|---|---|
| **K-Means** | Хорошо разделяет три группы монстров (рядовые / элита / боссы). Быстрый, легко интерпретируемый. |
| **DBSCAN** | Обнаруживает аномальные записи (шум). Устойчив к выбросам, но чувствителен к параметрам eps и min_samples. |
| **Agglomerative** | Дендрограмма наглядно показывает иерархию сложности противников. Метрики схожи с K-Means. |

### Классификация

| Метод | Вывод |
|---|---|
| **Decision Tree** | Легко интерпретируемые правила, хорошая точность. Визуализация дерева позволяет объяснить логику ИС. |
| **Naive Bayes** | Самый быстрый метод, приемлемая точность. Предоставляет вероятностные оценки для каждого класса. |
| **Random Forest** | Лучшая точность и устойчивость. Важность признаков показывает, что HP и damage_avg наиболее значимы. |

### Применение в дипломной работе

Результаты анализа могут быть использованы в ИС «НРИ-Помощник» для:
- **Автоматической оценки сложности** составляемого боя по параметрам NPC;
- **Рекомендаций гейммастеру** о балансе группы и подбираемых противников;
- **Обнаружения аномалий** в импортируемых данных бестиария.

---
*Ноутбук создан в рамках выпускной квалификационной работы*  
*Направление: Прикладная информатика*